In [8]:
from __future__ import annotations

from pathlib import Path
import random
import time

import pandas as pd

try:
    from nba_api.stats.endpoints import playbyplayv3
except Exception as exc:
    raise RuntimeError(
        "Could not import nba_api PlayByPlayV3. Install with: pip install nba_api pandas"
    ) from exc

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

"Imports loaded."

GAME_ID = "0021201216"  # Example: regular season game id
SEASON = "2012-13"      # Used only for season-file target path
PHASE = "regular"       # "regular" or "playoffs"

REQUEST_TIMEOUT = 12
RETRIES = 2
BACKOFF_BASE_SECONDS = 0.5

SAVE_ONE_OFF = True
ONE_OFF_DIR = Path("/Users/robschoen/Dropbox/CC/GLA/data/pbp/raw_backfill")

APPEND_TO_SEASON_FILE = False
NBA_DATA_REPO = Path("/Users/robschoen/Dropbox/CC/NBA_Data")

if PHASE not in {"regular", "playoffs"}:
    raise ValueError("PHASE must be 'regular' or 'playoffs'.")

GAME_ID

PBPV3_CANONICAL_COLUMNS = [
    "actionNumber",
    "clock",
    "period",
    "teamId",
    "teamTricode",
    "personId",
    "playerName",
    "playerNameI",
    "xLegacy",
    "yLegacy",
    "shotDistance",
    "shotResult",
    "isFieldGoal",
    "scoreHome",
    "scoreAway",
    "pointsTotal",
    "location",
    "description",
    "actionType",
    "subType",
    "videoAvailable",
    "shotValue",
    "actionId",
    "gameId",
]


def normalize_game_id(game_id: object) -> str:
    if pd.isna(game_id):
        return ""
    gid = str(game_id).strip()
    if gid.endswith(".0"):
        gid = gid[:-2]
    digits = "".join(ch for ch in gid if ch.isdigit())
    if not digits:
        return gid
    return digits.zfill(10)


def normalize_api_pbpv3_df(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    for col in PBPV3_CANONICAL_COLUMNS:
        if col not in d.columns:
            d[col] = pd.NA

    d = d[PBPV3_CANONICAL_COLUMNS]
    d["gameId"] = d["gameId"].map(normalize_game_id)
    d = d[d["gameId"] != ""].copy()

    int_default_zero_cols = [
        "actionNumber",
        "period",
        "teamId",
        "personId",
        "xLegacy",
        "yLegacy",
        "shotDistance",
        "isFieldGoal",
        "pointsTotal",
        "videoAvailable",
        "shotValue",
    ]
    for col in int_default_zero_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0).astype("int64")

    d["scoreHome"] = pd.to_numeric(d["scoreHome"], errors="coerce")
    d["scoreAway"] = pd.to_numeric(d["scoreAway"], errors="coerce")

    action_id_numeric = pd.to_numeric(d["actionId"], errors="coerce")
    d["actionId"] = action_id_numeric.fillna(d["actionNumber"]).astype("int64")

    d = d.sort_values(["gameId", "actionNumber"], kind="stable").reset_index(drop=True)
    return d


def season_target_path(season: str, phase: str, repo_dir: Path) -> Path:
    start_year = int(season.split("-")[0])
    if phase == "regular":
        filename = f"api_pbpv3_{start_year}.csv"
    else:
        filename = f"api_pbpv3_po_{start_year}.csv"
    return repo_dir / "PBPdata" / "api_pbpv3" / phase / filename


print(f"Canonical columns: {len(PBPV3_CANONICAL_COLUMNS)}")

def fetch_single_game_pbp(game_id: str, timeout: float, retries: int, backoff_base_seconds: float) -> pd.DataFrame:
    game_id = normalize_game_id(game_id)
    if len(game_id) != 10:
        raise ValueError(f"Expected a 10-digit game id, got: {game_id!r}")

    attempt_count = max(1, int(retries))
    errors: list[str] = []

    for attempt in range(1, attempt_count + 1):
        try:
            response = playbyplayv3.PlayByPlayV3(game_id=game_id, timeout=timeout)
            dfs = response.get_data_frames()
            if dfs and dfs[0] is not None and not dfs[0].empty:
                out = dfs[0].copy()
                if "gameId" not in out.columns:
                    out["gameId"] = game_id
                return out
            errors.append(f"attempt {attempt}: empty response")
        except Exception as exc:
            errors.append(f"attempt {attempt}: {type(exc).__name__}: {exc}")

        if attempt < attempt_count:
            sleep_seconds = backoff_base_seconds * (2 ** (attempt - 1)) + random.uniform(0.0, 0.35)
            time.sleep(sleep_seconds)

    raise RuntimeError("PlayByPlayV3 fetch failed. " + " | ".join(errors[-4:]))


raw_df = fetch_single_game_pbp(
    game_id=GAME_ID,
    timeout=REQUEST_TIMEOUT,
    retries=RETRIES,
    backoff_base_seconds=BACKOFF_BASE_SECONDS,
)

pbp_df = normalize_api_pbpv3_df(raw_df)

print(f"Fetched {len(raw_df):,} raw rows and {len(pbp_df):,} normalized rows for game {normalize_game_id(GAME_ID)}")
pbp_df.head(8)

summary = {
    "game_id": pbp_df["gameId"].iloc[0],
    "row_count": int(len(pbp_df)),
    "periods": sorted(pbp_df["period"].dropna().astype(int).unique().tolist()),
    "duplicate_action_numbers": int(pbp_df["actionNumber"].duplicated().sum()),
    "max_score_home": None if pbp_df["scoreHome"].dropna().empty else int(pbp_df["scoreHome"].dropna().max()),
    "max_score_away": None if pbp_df["scoreAway"].dropna().empty else int(pbp_df["scoreAway"].dropna().max()),
}
summary

pbp_df[["period", "clock", "actionNumber", "actionType", "subType", "description", "scoreHome", "scoreAway"]].tail(15)

gid = normalize_game_id(GAME_ID)

if SAVE_ONE_OFF:
    ONE_OFF_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = ONE_OFF_DIR / f"api_pbpv3_{gid}.csv"
    json_path = ONE_OFF_DIR / f"api_pbpv3_{gid}.json"

    pbp_df.to_csv(csv_path, index=False)
    pbp_df.to_json(json_path, orient="records")

    print(f"Wrote CSV:  {csv_path}")
    print(f"Wrote JSON: {json_path}")
else:
    print("SAVE_ONE_OFF is False. Skipping one-off export.")


Canonical columns: 24


RuntimeError: PlayByPlayV3 fetch failed. attempt 1: ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=12) | attempt 2: ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=12)